In [1]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader
import winsound
import itertools
import random


# grid init

In [59]:
custom_pretrained='original' #original
kind = 'patches_224'  # Example kind, can be changed
selected_FE = 'clip-vit-large-patch14-inter'
#'clip-vit-large-patch14-inter'# #'trocr-base-stage1'#'clip-vit-large-patch14'#'DeiT-Tiny' 
# #'clip-vit-large-patch14'  # Example feature extractor, can be changed
if custom_pretrained=='original':
    save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\torch_model_trained_on_rep\\'
else:
    save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\{custom_pretrained}\\torch_model_trained_on_rep\\'

clear_directory = True  # Set to True to clear the directory before saving checkpoints
script_name = source_path+"/scripts/torch_train_on_rep.py"
extra_view=False
extra_integration_mode = 'concat'  # 'concat' or 'add'
data_augmentation = False

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
search_type = 'single_search'#'grid_search'  # or 'random_search'
model_list = ['MLPClassifier1','MLPClassifier2','MLPClassifier3']
best_result = [np.inf, np.inf,np.inf]
all_results = []
grid_search_params = {
    'lr': [1e-4,1e-3,1e-2],
    'dropout': [0.2, 0.4, 0.7],
    'n_neurons': [32, 64, 128, 256, 512],
    'model_name': ['MLPClassifier1','MLPClassifier2'],
    'optimizer': ['Adam','AdamW'],
    'scheduler': ['no_scheduling'],
    'log_grad_norm': [True, False],
    'activation': ['relu'],
    'with_input_norm': ['dataset_norm','batch_norm',None],  # Whether to use input normalization
}
random_search_params = {    
    'lr': [1e-6,1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'dropout': [0.1, 0.2, 0.4, 0.7,0.9],
    'n_neurons': [16, 32 , 64, 128, 256, 512],
    'model_name': ['MLPClassifier1', 'MLPClassifier2'],
    'optimizer': ['Adam','AdamW','SGD'],
    'scheduler': ['no_scheduling','OneCycleLR','CosineAnnealingLR','CyclicalLR', 'ReduceLROnPlateau','StepLR','CosineAnnealingWarmRestarts'],
    'log_grad_norm': [True, False],
    'activation': ['relu', 'tanh','leaky_relu'],
    'with_input_norm': [True,False],
}
#{'weight_decay': weight_decay}
#'no_scheduling'
#'CosineAnnealingLR' {'T_max': total_epochs, 'eta_min': lr_final}
#'OneCycleLR' {'total_epochs': total_epochs, 'steps_per_epoch': 705, 'max_lr':0.1}
# 'CosineAnnealingWarmRestarts'
single_experiment = {
    'lr': [1e-5],
    'dropout': [0.1],
    'n_neurons': [128], #not used if hiddden_sizes is used
    'hidden_sizes': [[16]],
    'model_name': ['MLPClassifier1'],
    'optimizer': ['AdamW'],
    'scheduler': ['no_scheduling'],
    'log_grad_norm': [True],
    'activation': ['relu'],
    'with_input_norm': ['batch_norm'],#['batch_norm'],
}
if search_type == 'grid_search':
    param_grid = grid_search_params
elif search_type == 'random_search':
    param_grid = random_search_params
else:
    param_grid = single_experiment

keys, values = zip(*param_grid.items())
all_combos = list(itertools.product(*values))

# Shuffle combinations
random.shuffle(all_combos)
if search_type == 'random_search': 
    # Pick N random samples (e.g., 5)
    N = 100 if 100 < len(all_combos) else len(all_combos)
    experiments = all_combos[:N]
else:
    # For grid search, use all combinations
    experiments = all_combos[:]

In [60]:
prev_log = os.path.join(save_common, f'{search_type}_results.csv')
if os.path.exists(prev_log) and search_type != 'single_search':
    prev_results = pd.read_csv(prev_log)
    print(prev_results['best_val_loss'].min())
    print(prev_results.iloc[prev_results['best_val_loss'].idxmin()])
    #display(prev_results.iloc[prev_results['best_val_loss'].idxmin()])
    #print(prev_results['id'])
    for model in model_list:
        model_results = prev_results[prev_results['model_name'] == model]
        if not model_results.empty:
            best_result_temp = model_results['best_val_loss'].min()
            print(f"Best result for {model}: {best_result_temp}")
            best_result[model_list.index(model)]= best_result_temp
        else:
            print(f"No results found for model: {model}")
    last_index = prev_results['id'].max() if not prev_results.empty else 0
else:
    prev_results = None
    last_index = None
print(best_result)
print(f"Last index: {last_index}")
print(search_type)
print(len(experiments))
print(experiments)

[inf, inf, inf]
Last index: None
single_search
1
[(1e-05, 0.1, 128, [16], 'MLPClassifier1', 'AdamW', 'no_scheduling', True, 'relu', 'batch_norm')]


# run

In [61]:
suffix = '_augmented' if data_augmentation else ''
train_filename,val_filename, _ = file_IO.load_input_files(source_path,selected_FE,kind,suffix, custom_pretrained=custom_pretrained)
if extra_view:
    extra_train_filename, extra_val_filename = file_IO.load_input_files(source_path,selected_FE,kind='body',suffix='', custom_pretrained=custom_pretrained)
else:
    extra_train_filename, extra_val_filename = None, None

loss_criterion = 'CrossEntropyLoss'
total_epochs = 100
use_profiler = False
profiler_config = None
plot_every = 1
patience = 2
run_epochs = total_epochs
use_amp = False
val_percentage = 1.0
batch_size = 64
aggregation_mode = None  # 'mean' or 'max'
weight_decay = 1e-2  # Weight decay for the optimizer 
lr_final = 1e-8  # Final learning rate for the backbone

# Assign unique IDs
start = last_index + 1 if last_index is not None else 0
for i, combo in enumerate(experiments[start:]):
    experiment_dict = dict(zip(keys, combo))
    '''if experiment_dict['model_name']== 'MLPClassifier2' and experiment_dict['n_neurons'] >= 128:
        continue
    if experiment_dict['model_name']== 'MLPClassifier1' and  experiment_dict['with_input_norm'] == None:
        continue'''
    experiment_dict['id'] = i+start
    ##################################################
    model_name = experiment_dict['model_name']
    save_path = save_common+f'{model_name}'
    file_IO.access_or_create_dir(save_path)
    checkpoint_path=save_path+'\\checkpoints'
    file_IO.access_or_create_dir(checkpoint_path)
    if clear_directory==True:
        print(f'Clearing directory: {checkpoint_path}')
        file_IO.clear_folder(checkpoint_path)

    log_grad_norm = experiment_dict['log_grad_norm']
    lr = experiment_dict['lr']  # Learning rate for the optimizer
    with_input_norm = experiment_dict.get('with_input_norm', None)  # Whether to use input normalization
    #for full fine tuning
    optimizer_phases = [total_epochs]  
    optim_config = {
        'optimizer_phases':optimizer_phases,  
        'type_of_training': 'from_scratch',
        'scheduling': experiment_dict['scheduler'],  
        'optimizer_name':experiment_dict['optimizer'], 
        'phase_lr': [lr],
        'phase_optimizer_hyperparams': [{'weight_decay': weight_decay}],
        'phase_scheduler_hyperparams': [{'T_max': total_epochs, 'eta_min': lr_final, 'total_epochs': total_epochs,
                                          'steps_per_epoch': 705, 'max_lr':0.01,'step_size': 10,'patience':int(patience/2),
                                          'max_lr_cycle':0.001, 'base_lr_cycle': 0.0001}],
    }
    if optim_config['scheduling'][0] in ['OneCycleLR','CyclicalLR']:
        step_at_epoch = True
    else:
        step_at_epoch = False
    args = script_launching.DotDict(
        data_augmentation=data_augmentation,
        extra_view=extra_view,
        loss_criterion=loss_criterion,
        model_name=model_name,
        total_epochs=total_epochs,
        patience=patience,
        log_grad_norm=log_grad_norm,
        use_amp = use_amp,
        batch_size=batch_size,
        val_percentage=val_percentage,
        weight_decay=weight_decay,
        lr=lr,
        lr_final=lr_final,
        optim_config=optim_config,
        aggregation_mode=aggregation_mode,
        extra_integration_mode=extra_integration_mode,
        train_filename=train_filename,
        val_filename=val_filename,
        extra_train_filename=extra_train_filename,
        extra_val_filename=extra_val_filename,
        step_at_epoch=step_at_epoch,
        experiment_id=experiment_dict['id'],
        with_input_norm=with_input_norm,
    )
    file_IO.save_args(args,checkpoint_path)  # Save the arguments to a file
    ######################################
    # Define datasets and group by page
    train_df = pd.read_csv(train_filename)
    val_df = pd.read_csv(val_filename)
    #print(len(train_df.columns))
    #cols_to_drop = [c for c in val_df.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()][256:]
    #train_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    #val_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    #print(len(train_df.columns))
    mean,scale = model_utils.get_normalization_parameters(train_filename)

    if extra_view:
        train_df_extra = pd.read_csv(extra_train_filename)
        val_df_extra = pd.read_csv(extra_val_filename)
        train_df = dataframes.merge_dfs(train_df, train_df_extra, mode=extra_integration_mode)
        val_df = dataframes.merge_dfs(val_df, val_df_extra, mode=extra_integration_mode)

    train_df = dataframes.aggregate_dfs(train_df,mode=aggregation_mode)
    val_df = dataframes.aggregate_dfs(val_df,mode=aggregation_mode)
    #train_df=file_IO.change_filename_from_to(train_df, fr=saved, to=running
    #cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
    cols_to_keep = [c for c in train_df.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()]
    in_features = len(cols_to_keep)  # Number of features from the model output

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device is: ",device)

    train_dataset = dataframes.CustomExtractedDataset(train_df, label_column='male')
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = dataframes.CustomExtractedDataset(val_df, label_column='male')
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    print(f"[GPU Memory] Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB | Reserved: {torch.cuda.memory_reserved() / 1e6:.2f} MB")

    if 'hidden_sizes' in experiment_dict:
        kwargs = {'hidden_sizes': experiment_dict['hidden_sizes']}
    else:
        kwargs = {}
    loss_fn = training_utils.get_criterion(name=loss_criterion)
    model = model_utils.get_classification_head(name=model_name, in_features=in_features, num_classes=2,
                                                dropout=experiment_dict['dropout'], n_neurons=experiment_dict['n_neurons'],
                                                activation=experiment_dict['activation'],with_input_norm=with_input_norm, 
                                                mean=mean,scale=scale,**kwargs)
    print(model)

    best_model_performance=training_utils.train_fine(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        device=device,
        total_epochs=total_epochs,
        loss_fn=loss_fn,
        use_profiler=use_profiler,
        profiler_config=profiler_config,
        save_path=save_path,
        plot_every=plot_every,
        early_stopping_patience=patience,
        checkpoint_path=checkpoint_path+"\\checkpoint.pt",
        log_grad_norm=log_grad_norm,
        run_epochs=run_epochs,
        use_amp=use_amp,
        val_percentage=val_percentage,  # Use 10% of validation data for linear evaluation
        optim_config=optim_config,  # e.g., 'Adam', 'SGD', 'AdamW'
        save_backbone=False,
        step_at_epoch=step_at_epoch,
        # ... other parameters
    )

    for key in experiment_dict.keys():
        if key not in best_model_performance:
            best_model_performance[key] = experiment_dict[key]
    all_results.append(best_model_performance)

    index=model_list.index(model_name)
    if best_model_performance['best_val_loss'] < best_result[index]:
        best_checkpoint = os.path.join(checkpoint_path, "checkpoint_best.pt")
        destination = os.path.join(save_path, "checkpoint_best.pt")
        shutil.copy2(best_checkpoint, destination)
        best = os.path.join(checkpoint_path, "training_plot.png")
        destination = os.path.join(save_path, "training_plot.png")
        shutil.copy2(best, destination)
        best = os.path.join(checkpoint_path, "args.txt")
        destination = os.path.join(save_path, "args.txt")
        shutil.copy2(best, destination)
        best_result[index] = best_model_performance['best_val_loss']

    all_results_df = pd.DataFrame(all_results)
    if prev_results is not None:
        all_results_df = pd.concat([prev_results, all_results_df], ignore_index=True)
    all_results_df.to_csv(os.path.join(save_common, f'{search_type}_results.csv'), index=False)
    '''duration = 3000  # milliseconds
    freq = 880  # Hz
    winsound.Beep(freq, duration)'''

Clearing directory: c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14-inter\representation_extraction\torch_model_trained_on_rep\MLPClassifier1\checkpoints
aggregating patches, length before: 45120
length after: 45120
aggregating patches, length before: 11360
length after: 11360
Device is:  cuda
Extracted 1024 feature columns:
Extracted 1024 feature columns:
[GPU Memory] Allocated: 17.16 MB | Reserved: 31.46 MB
CustomMLP(
  (model): Sequential(
    (0): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (1): Linear(in_features=1024, out_features=16, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.1, inplace=False)
    (4): Linear(in_features=16, out_features=2, bias=True)
  )
)
2025-11-16 23:21:30,650 - INFO - Model size: 0.09 MB
📂 No checkpoint found at c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14-inter\re

Epoch 1/100 [Val]: 100%|██████████| 178/178 [00:00<00:00, 186.35it/s]

2025-11-16 23:21:37,384 - INFO - Epoch 1| Train Accuracy 0.6226| Train Loss: 0.6477 | Val Acc: 0.6436 | Val Loss: 0.6287 | Avg Grad Norm: 1.2764 | Epoch Time: 6.72s | Val Time: 0.98s
2025-11-16 23:21:37,387 - INFO - block 0 lr: 0.000010
2025-11-16 23:21:37,395 - INFO - ✅ Saved new best model at epoch 1
2025-11-16 23:21:37,400 - INFO - ⏳ No improvement for 0 epoch(s)


2025-11-16 23:21:38,253 - INFO - 
Epoch 1 - Optimization Phase: 0


Epoch 2/100 [Val]: 100%|██████████| 178/178 [00:00<00:00, 223.79it/s]

...

2025-11-16 23:22:49,761 - INFO - Epoch 11| Train Accuracy 0.7649| Train Loss: 0.4884 | Val Acc: 0.7060 | Val Loss: 0.5603 | Avg Grad Norm: 1.3731 | Epoch Time: 6.96s | Val Time: 1.06s
2025-11-16 23:22:49,761 - INFO - block 0 lr: 0.000010
2025-11-16 23:22:49,769 - INFO - ⏳ No improvement for 1 epoch(s)


2025-11-16 23:22:50,689 - INFO - 
Epoch 11 - Optimization Phase: 0


Epoch 12/100 [Val]: 100%|██████████| 178/178 [00:00<00:00, 191.14it/s]

2025-11-16 23:22:57,352 - INFO - Epoch 12| Train Accuracy 0.7685| Train Loss: 0.4836 | Val Acc: 0.7076 | Val Loss: 0.5569 | Avg Grad Norm: 1.3696 | Epoch Time: 6.66s | Val Time: 0.94s
2025-11-16 23:22:57,354 - INFO - block 0 lr: 0.000010
2025-11-16 23:22:57,360 - INFO - ⏳ No improvement for 2 epoch(s)
2025-11-16 23:22:57,360 - INFO - ⛔ Early stopping at epoch 12 (no improvement for 2 epochs)



📊 Performance Summary:
Average batch time: 0.0044s
Peak GPU memory usage: 17.47 MB
9


# reload

In [23]:
def reload_modules():
    import importlib
    import utils.data_loading as data_loading
    import utils.visualization as visualization
    import utils.dataframes as dataframes
    import utils.utils_transforms as u_transforms
    import utils.training_utils as training_utils
    import utils.model_utils as model_utils
    import utils.file_IO as file_IO
    import utils.vit_rollout_mod as vit_rollout_mod
    import utils.script_launching as script_launching
    
    importlib.reload(file_IO)
    importlib.reload(data_loading)
    importlib.reload(visualization)
    importlib.reload(dataframes)
    importlib.reload(u_transforms)
    importlib.reload(model_utils)
    importlib.reload(training_utils)
    importlib.reload(vit_rollout_mod)
    importlib.reload(script_launching)

    return data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching
data_loading, visualization, dataframes, u_transforms, training_utils, model_utils, file_IO, vit_rollout_mod, script_launching = reload_modules()